### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\dell\AppData\Local\Temp\ipykernel_5848\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\dell\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Information Security (IS) Mid Term Spring 2024.pdf
  ✓ Loaded 3 pages

Processing: Legacyloop fyp Phase 1 Final  (7).pdf.pdf
  ✓ Loaded 31 pages

Total documents loaded: 34


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2024-12-11T19:23:46+05:00', 'author': 'Muhammad Huzaifa .', 'moddate': '2024-12-11T19:23:46+05:00', 'source': '..\\data\\pdfs\\Information Security (IS) Mid Term Spring 2024.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Information Security (IS) Mid Term Spring 2024.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2024-12-11T19:23:46+05:00', 'author': 'Muhammad Huzaifa .', 'moddate': '2024-12-11T19:23:46+05:00', 'source': '..\\data\\pdfs\\Information Security (IS) Mid Term Spring 2024.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'source_file': 'Information Security (IS) Mid Term Spring 2024.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2024

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 34 documents into 80 chunks

Example chunk:
Content: U n i v e r s i t y  o f  C e n t r a l  P u n j a b  
BSCS FINAL PROJECT  
Software Requirements Specification  
L e g a c y L o o p
 
P r o d u c t  O w n e r  
 
G r o u p  I D :  x x x x x x x  
S...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-05-19T20:42:10+00:00', 'title': 'Legacyloop fyp Phase 1 Final  (7).pdf', 'moddate': '2026-05-19T20:42:10+00:00', 'keywords': 'DAHKJPtsB7w,BAGn-i8XaQo,0', 'author': 'Alishba Niaz', 'trapped': '/False', 'source': '..\\data\\pdfs\\Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-05-19T20:42:10+00:00', 'title': 'Legacyloop fyp Phase 1 Final  (7).pdf', 'moddate': '2026-05-19T20:42:10+00:00', 'keywords': 'DAHKJPtsB7w,BAGn-i8XaQo,0', 'author': 'Alishba Niaz', 'trapped': '/False', 'source': '..\\data\\pdfs\\Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'file_type': 'pdf'}, page_content='U n i v e r s i t y  o f  C e n t r a l  P u n j a b  \nBSCS FINAL PROJECT  \nSoftware Requirements Specification  \nL e g a c y L o o p\n \nP r o d u c t  O w n e r  \n \nG r o u p  I D :  x x x x x x x  \nS t u d e n t  R e g #  \nP r e s e n t e d b y :\n< S u p e r v i s o r  N a m e >  \nS t u d e n t  N a m e  \nF a c u l t y  o f  I n f o r m a t i o n  T e c h n o l o g y  &  C o m p u t e r  S c i e n c e  \nU n i v e r s i t y  o f  C e n t r a l  P u n j a b'),
 Document(metadata={'pro

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3299.12it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\dell\AppData\Local\Temp\ipykernel_5848\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 80


In [9]:
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-05-19T20:42:10+00:00', 'title': 'Legacyloop fyp Phase 1 Final  (7).pdf', 'moddate': '2026-05-19T20:42:10+00:00', 'keywords': 'DAHKJPtsB7w,BAGn-i8XaQo,0', 'author': 'Alishba Niaz', 'trapped': '/False', 'source': '..\\data\\pdfs\\Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'Legacyloop fyp Phase 1 Final  (7).pdf.pdf', 'file_type': 'pdf'}, page_content='U n i v e r s i t y  o f  C e n t r a l  P u n j a b  \nBSCS FINAL PROJECT  \nSoftware Requirements Specification  \nL e g a c y L o o p\n \nP r o d u c t  O w n e r  \n \nG r o u p  I D :  x x x x x x x  \nS t u d e n t  R e g #  \nP r e s e n t e d b y :\n< S u p e r v i s o r  N a m e >  \nS t u d e n t  N a m e  \nF a c u l t y  o f  I n f o r m a t i o n  T e c h n o l o g y  &  C o m p u t e r  S c i e n c e  \nU n i v e r s i t y  o f  C e n t r a l  P u n j a b'),
 Document(metadata={'pro

### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("Functional Requirements of legacy loop")

Retrieving documents for query: 'Functional Requirements of legacy loop'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_411c1cce_55',
  'content': 'S 2 6 C S 0 4 3  \nL e g a c y  L o o p  \nF Y P  P h a s e  I  ( R S )  P a g e  1 8  \n5 . 2  A c c e p t a n c e  C r i t e r i a  \nThe acceptance criteria for Legacy Loop define the conditions that must be satisfied for each feature\nto be considered successfully implemented and ready for deployment. These criteria are based on the\ndefined user stories and functional requ irements of the system. Each feature will be tested through\nspecific test scenarios to verify correct functionality, expected behavior, u sability, and system\nreliability.  \nUser enters valid registration\ndetails and submits form  \nUser enters valid email and\npassword  \nUser enters incorrect credentials  \nUser uploads book/resource\ndetails with image  \nUser searches resources using\nkeywords or course tags  \nMentor creates tutoring\nadvertisement  \nStudent searches available\nmentors  \nUser sends message to another\nuser  \nUser submits syllabus-related\nacad

In [14]:
rag_retriever.retrieve("5 main features of legacy loop")


Retrieving documents for query: '5 main features of legacy loop'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_411c1cce_55',
  'content': 'S 2 6 C S 0 4 3  \nL e g a c y  L o o p  \nF Y P  P h a s e  I  ( R S )  P a g e  1 8  \n5 . 2  A c c e p t a n c e  C r i t e r i a  \nThe acceptance criteria for Legacy Loop define the conditions that must be satisfied for each feature\nto be considered successfully implemented and ready for deployment. These criteria are based on the\ndefined user stories and functional requ irements of the system. Each feature will be tested through\nspecific test scenarios to verify correct functionality, expected behavior, u sability, and system\nreliability.  \nUser enters valid registration\ndetails and submits form  \nUser enters valid email and\npassword  \nUser enters incorrect credentials  \nUser uploads book/resource\ndetails with image  \nUser searches resources using\nkeywords or course tags  \nMentor creates tutoring\nadvertisement  \nStudent searches available\nmentors  \nUser sends message to another\nuser  \nUser submits syllabus-related\nacad